In [5]:
conda install pandas

3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.




==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda




In [9]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Load dataset
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values / 255.0
y = mnist.target.astype(int).values.reshape(-1,1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

# Architecture
input_size = 784
hidden1 = 128
hidden2 = 64
output_size = 10

# He initialization
W1 = np.random.randn(input_size, hidden1) * np.sqrt(2/input_size)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2/hidden1)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size) * np.sqrt(2/hidden2)
b3 = np.zeros((1, output_size))

learning_rate = 0.01
epochs = 20
batch_size = 64

# Training
for epoch in range(epochs):

    for i in range(0, X_train.shape[0], batch_size):

        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        m = X_batch.shape[0]

        # Forward
        Z1 = X_batch @ W1 + b1
        A1 = relu(Z1)

        Z2 = A1 @ W2 + b2
        A2 = relu(Z2)

        Z3 = A2 @ W3 + b3
        A3 = softmax(Z3)

        # Loss
        loss = -np.mean(np.sum(y_batch*np.log(A3+1e-8), axis=1))

        # Backprop
        dZ3 = A3 - y_batch
        dW3 = A2.T @ dZ3 / m
        db3 = np.sum(dZ3, axis=0, keepdims=True) / m

        dA2 = dZ3 @ W3.T
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = A1.T @ dZ2 / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = X_batch.T @ dZ1 / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Update
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    print("Epoch", epoch+1, "Loss:", loss)

# Testing
Z1 = X_test @ W1 + b1
A1 = relu(Z1)

Z2 = A1 @ W2 + b2
A2 = relu(Z2)

Z3 = A2 @ W3 + b3
A3 = softmax(Z3)

pred = np.argmax(A3, axis=1)
true = np.argmax(y_test, axis=1)

accuracy = np.mean(pred == true)

print("Test Accuracy:", accuracy)

Epoch 1 Loss: 0.5391623295346141
Epoch 2 Loss: 0.41111052342821536
Epoch 3 Loss: 0.343837833687465
Epoch 4 Loss: 0.2928616971789153
Epoch 5 Loss: 0.24913430037907736
Epoch 6 Loss: 0.21544212680078967
Epoch 7 Loss: 0.19148705154129642
Epoch 8 Loss: 0.17357871840906935
Epoch 9 Loss: 0.15937378957960718
Epoch 10 Loss: 0.14657779391800965
Epoch 11 Loss: 0.1354361981039517
Epoch 12 Loss: 0.12467618167984698
Epoch 13 Loss: 0.11600854753840799
Epoch 14 Loss: 0.10826169913831152
Epoch 15 Loss: 0.10125110841753174
Epoch 16 Loss: 0.09516415822807178
Epoch 17 Loss: 0.0900805782504219
Epoch 18 Loss: 0.08537893878952736
Epoch 19 Loss: 0.08127069601069364
Epoch 20 Loss: 0.07806744651543962
Test Accuracy: 0.9619285714285715
